# RetailPulse — Guided Analytics Walkthrough

This notebook explains the pipeline in a step-by-step way. Run it from the project root. It uses the downloaded UCI workbook when present; otherwise it uses the clearly labelled synthetic sample.

## 1. Set up and choose the input

The real UCI dataset is downloaded separately with `python scripts/download_data.py`. The sample exists for offline learning and testing.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('../').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from retailpulse.analysis import read_source, prepare_transactions, calculate_analytics
real_file = ROOT / 'data/raw/online_retail_II.xlsx'
sample_file = ROOT / 'data/sample/sample_retail_transactions.csv'
input_file = real_file if real_file.exists() else sample_file
print('Input:', input_file)
print('Synthetic demo sample:', input_file == sample_file)

## 2. Inspect the raw structure

Start with shape, column names, and a few records. This is a basic data-understanding step before cleaning.

In [ ]:
raw = read_source(input_file)
print('Rows, columns:', raw.shape)
print(raw.dtypes)
display(raw.head())

## 3. Clean and classify transactions

We standardise the source schema, parse dates and numeric fields, and mark sales versus returns/cancellations. Customer IDs may be missing; those rows can still contribute to sales totals.

In [ ]:
prepared, quality = prepare_transactions(raw)
print(quality)
display(prepared[['invoice','stock_code','quantity','unit_price','line_value','is_sale','is_return_or_cancellation']].head())

## 4. Calculate core business metrics

Gross sales exclude cancellation invoices and negative-quantity lines. Return/cancellation value is reported separately rather than silently netted.

In [ ]:
result = calculate_analytics(prepared, quality)
result['kpis']

## 5. Sales and product performance

In [ ]:
display(result['monthly'].head(12))
display(result['products'].head(10))

## 6. Customer segmentation and cohort retention

RFM means Recency, Frequency, Monetary value. Cohorts are grouped by the month in which each customer first purchased.

In [ ]:
display(result['segment_summary'])
display(result['cohort_retention'].head())

## 7. Interpretation checklist

- Revenue is not profit: there is no cost or margin field.
- Customer-level outputs omit transactions without customer IDs.
- Recent cohorts have less time to repeat purchase (right-censoring).
- RFM segment names are heuristics, not validated labels.
- Do not present synthetic sample findings as results from the UCI dataset.